In [1]:
import os, sys
# nbconvert runs with cwd = notebooks/; the src modules (and this notebook's
# own path references below) assume the project root instead.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

# 02 — Modeling

Baseline logistic regression, then a tuned XGBoost, compared honestly against each other and against a majority-class baseline. The split happens before any preprocessing, and preprocessing lives inside a `Pipeline`, so scaling/encoding never sees the test fold -- the standard leakage mistake is avoided structurally.

In [2]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from sklearn.dummy import DummyClassifier
from src.data import load_clean
from src.model import split, fit_logreg, fit_xgb, evaluate

## Split first, then build the pipeline

In [3]:
df = load_clean('data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
X_train, X_test, y_train, y_test = split(df)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print(f'Train churn rate: {y_train.mean():.4f}   Test churn rate: {y_test.mean():.4f}')

Train: (5634, 19)  Test: (1409, 19)
Train churn rate: 0.2654   Test churn rate: 0.2654


`stratify=y` in the split keeps both folds at the same ~26.5% churn rate -- without it, the test-set churn rate wanders and the metrics below get noisy.

## Majority-class baseline

In [4]:
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
acc = (dummy.predict(X_test) == y_test).mean()
print(f'Accuracy: {acc:.4f}  (recall on churn class: 0.00)')

Accuracy: 0.7346  (recall on churn class: 0.00)


**A model that predicts "no churn" for every customer scores 73.5% accuracy while catching zero churners.** Accuracy is therefore not a usable metric here. The rest of this notebook reports precision/recall/ROC-AUC instead, and the operating threshold is chosen separately against business cost in notebook 03 -- not fixed at the default 0.5.

## Logistic regression baseline

In [5]:
logreg = fit_logreg(X_train, y_train)
logreg_results = evaluate(logreg, X_test, y_test, name='Logistic Regression')


=== Logistic Regression (threshold=0.5) ===
              precision    recall  f1-score   support

        Stay       0.90      0.72      0.80      1035
       Churn       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.8415
PR-AUC : 0.6329
Confusion matrix:
 [[747 288]
 [ 81 293]]


## XGBoost, tuned via grid search

In [6]:
search = fit_xgb(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV ROC-AUC:', round(search.best_score_, 4))
best_xgb = search.best_estimator_

Best params: {'clf__colsample_bytree': 0.8, 'clf__max_depth': 3, 'clf__min_child_weight': 5, 'clf__subsample': 0.8}
Best CV ROC-AUC: 0.845


In [7]:
xgb_results = evaluate(best_xgb, X_test, y_test, name='XGBoost (tuned)')


=== XGBoost (tuned) (threshold=0.5) ===
              precision    recall  f1-score   support

        Stay       0.91      0.74      0.81      1035
       Churn       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409

ROC-AUC: 0.8416
PR-AUC : 0.658
Confusion matrix:
 [[762 273]
 [ 77 297]]


## Model comparison

In [8]:
table = pd.DataFrame([
    {'Model': 'Predict-majority baseline', 'CV ROC-AUC': 0.500, 'Test ROC-AUC': 0.500,
     'Recall (churn)': 0.00, 'Precision (churn)': None},
    {'Model': 'Logistic regression', 'CV ROC-AUC': None,
     'Test ROC-AUC': round(logreg_results['auc'], 3),
     'Recall (churn)': round(logreg_results['recall_churn'], 2),
     'Precision (churn)': round(logreg_results['precision_churn'], 2)},
    {'Model': 'XGBoost (tuned)', 'CV ROC-AUC': round(search.best_score_, 3),
     'Test ROC-AUC': round(xgb_results['auc'], 3),
     'Recall (churn)': round(xgb_results['recall_churn'], 2),
     'Precision (churn)': round(xgb_results['precision_churn'], 2)},
])
table

,Model,CV ROC-AUC,Test ROC-AUC,Recall (churn),Precision (churn)
0,Predict-majority baseline,0.500,0.500,0.00,NaN
1,Logistic regression,NaN,0.841,0.78,0.50
2,XGBoost (tuned),0.845,0.842,0.79,0.52


## Honest read of the comparison

Tuned XGBoost lands at **0.842 test ROC-AUC** versus **0.841** for plain logistic regression -- a 0.001 gap, well inside noise. Gradient boosting is not meaningfully better than the linear baseline on this dataset.

**Recommendation:** for a retention team that needs to explain every individual offer to a customer or a compliance reviewer, logistic regression's coefficients are directly interpretable and the interpretability cost of XGBoost is not justified by a 0.001 AUC gain. XGBoost is retained here for SHAP-based feature discovery in notebook 03, where its interaction terms surface a slightly richer feature ranking, but the model actually recommended for production is the simpler one unless product requirements change.

Models and the test split are saved to disk (via `python -m src.train`, run from the project root) for the explainability and threshold-optimization notebook that follows.